# 固态电解质高通量筛选案例研究
# Solid Electrolyte High-Throughput Screening Case Study

本Notebook演示如何筛选高Li离子电导率的硫化物固态电解质

**流程概览：**
1. 从Materials Project搜索候选材料
2. DFT结构优化与能量计算
3. 机器学习势训练
4. 多温度MD模拟
5. 离子电导率与活化能分析
6. 候选材料排名与可视化

## 1. 环境设置与导入

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# 导入自定义模块
sys.path.insert(0, '/root/.openclaw/workspace/dft_lammps_research')
sys.path.insert(0, '/root/.openclaw/workspace/dft_lammps_research/applications/solid_electrolyte')

from case_solid_electrolyte import (
    SolidElectrolyteConfig, SulfideElectrolyteScreener
)

# 科学计算库
from pymatgen.core import Structure, Composition
from ase import Atoms
from ase.units import kB, fs

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns

# 设置可视化样式
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ 环境设置完成")
print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 配置参数设置

In [ ]:
# 创建配置
config = SolidElectrolyteConfig(
    target_ion="Li",
    target_conductivity=1e-4,  # S/cm
    search_chemsys=["Li-S-P", "Li-Ge-S", "Li-Sn-S", "Li-P-S-Cl"],
    max_entries=50,
    md_temperatures=[300, 400, 500, 600, 700, 800, 900],
    work_dir="./solid_electrolyte_results",
)

print("筛选配置:")
print(f"  目标离子: {config.target_ion}")
print(f"  目标电导率: {config.target_conductivity:.0e} S/cm")
print(f"  搜索体系: {config.search_chemsys}")
print(f"  MD温度: {config.md_temperatures} K")
print(f"  工作目录: {config.work_dir}")

## 3. Phase 1: 从Materials Project搜索候选材料

In [ ]:
# 创建筛选器
screener = SulfideElectrolyteScreener(config)

# 搜索材料 (使用演示数据)
candidates = screener.search_materials_project()

print(f"\n找到 {len(candidates)} 个候选材料\n")

# 显示候选列表
df_candidates = pd.DataFrame(candidates)
df_candidates[['material_id', 'formula', 'energy_per_atom', 'band_gap', 'spacegroup']]

## 4. Phase 2: DFT计算

In [ ]:
# 运行DFT计算 (使用模拟数据)
dft_results = screener.run_dft_calculations(candidates, skip_dft=True)

# 显示DFT结果
print("\nDFT计算结果摘要:")
for result in dft_results:
    print(f"  {result['formula']}: E={result['dft_energy_per_atom']:.3f} eV/atom, "
          f"Eg={result['band_gap']:.2f} eV, B={result['bulk_modulus']:.1f} GPa")

## 5. Phase 3: 机器学习势训练

In [ ]:
# 训练ML势 (使用模拟)
ml_models = screener.train_ml_potentials(dft_results, skip_training=True)

print("\nML势训练结果:")
for formula, model_info in ml_models.items():
    print(f"  {formula}:")
    print(f"    训练损失: {model_info['training_loss']:.4f}")
    print(f"    验证损失: {model_info['validation_loss']:.4f}")
    print(f"    模型数量: {model_info['num_models']}")

## 6. Phase 4: 多温度MD模拟

In [ ]:
# 运行MD模拟
md_results = screener.run_md_simulations(dft_results, ml_models, skip_md=True)

print("\nMD模拟完成!\n")

# 显示一个材料的详细结果
example = md_results[0]
print(f"示例: {example['formula']}")
print(f"  活化能: {example['activation_energy']:.3f} eV")
print(f"\n  扩散系数 (cm²/s):")
for T, D in example['diffusion_coefficients'].items():
    print(f"    {T}K: {D:.2e}")
print(f"\n  离子电导率 (S/cm):")
for T, sigma in example['conductivities'].items():
    print(f"    {T}K: {sigma:.2e}")

## 7. Phase 5: 结果分析与排名

In [ ]:
# 分析结果
df_ranking = screener.analyze_results(md_results)

print("\n候选材料排名:")
print("="*80)
display(df_ranking[['Formula', 'Activation Energy (eV)', 'σ_300K (S/cm)', 
                   'Performance Score', 'Recommendation']])

## 8. 可视化分析

In [ ]:
# 生成可视化
screener.create_visualizations(md_results, df_ranking)

# 显示生成的图表
from IPython.display import Image, display

print("\n生成的图表:")
plot_path = Path(config.work_dir) / "solid_electrolyte_analysis.png"
if plot_path.exists():
    display(Image(filename=str(plot_path)))

## 9. 主图: 电导率-活化能散点图

In [ ]:
# 创建发表质量的电导率-活化能散点图
fig, ax = plt.subplots(figsize=(12, 9))

# 按性能评分着色
colors = []
for _, row in df_ranking.iterrows():
    score = row['Performance Score']
    if score > 3:
        colors.append('#2ecc71')  # 绿色
    elif score > 2:
        colors.append('#3498db')  # 蓝色
    elif score > 1:
        colors.append('#f39c12')  # 橙色
    else:
        colors.append('#e74c3c')  # 红色

# 绘制散点
scatter = ax.scatter(df_ranking['Activation Energy (eV)'], 
                    df_ranking['log10(σ_300K)'],
                    c=colors, s=400, alpha=0.8, 
                    edgecolors='black', linewidth=2)

# 添加标签
for _, row in df_ranking.iterrows():
    ax.annotate(row['Formula'], 
               (row['Activation Energy (eV)'], row['log10(σ_300K)']),
               xytext=(10, 10), textcoords='offset points',
               fontsize=11, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# 添加目标线
ax.axhline(y=-4, color='red', linestyle='--', alpha=0.7, linewidth=2,
          label='Target: σ = 10⁻⁴ S/cm')
ax.axvline(x=0.3, color='blue', linestyle='--', alpha=0.7, linewidth=2,
          label='Target: Ea = 0.3 eV')

# 添加理想区域
ax.fill_between([0, 0.3], [-4, -4], [0, 0], alpha=0.1, color='green',
               label='Target Region')

ax.set_xlabel('Activation Energy (eV)', fontsize=16, fontweight='bold')
ax.set_ylabel('log₁₀(σ) at 300K (S/cm)', fontsize=16, fontweight='bold')
ax.set_title('Solid Electrolyte Screening: Conductivity vs Activation Energy\n' + 
            f'({len(df_ranking)} Candidates Evaluated)', 
            fontsize=18, fontweight='bold', pad=20)
ax.legend(fontsize=13, loc='lower left')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=13)

plt.tight_layout()
plt.savefig(f"{config.work_dir}/fig_conductivity_vs_Ea_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ 散点图生成完成")

## 10. Arrhenius分析

In [ ]:
# 创建Arrhenius图
fig, ax = plt.subplots(figsize=(10, 7))

for i, result in enumerate(md_results):
    temps = result['md_temperatures']
    D_values = [result['diffusion_coefficients'].get(T, 0) for T in temps]
    
    inv_T = 1000 / np.array(temps)
    log_D = np.log(D_values)
    
    # 拟合
    coeffs = np.polyfit(inv_T, log_D, 1)
    Ea_fit = -coeffs[0] * 8.617e-5 * 1000  # 转换为eV
    
    ax.plot(inv_T, log_D, 'o-', label=f"{result['formula']} (Ea={Ea_fit:.2f}eV)",
           linewidth=2, markersize=8)

ax.set_xlabel('1000/T (K⁻¹)', fontsize=14, fontweight='bold')
ax.set_ylabel('ln(D) [cm²/s]', fontsize=14, fontweight='bold')
ax.set_title('Arrhenius Plot - Li⁺ Diffusion', fontsize=16, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.work_dir}/fig_arrhenius_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Arrhenius图生成完成")

## 11. 性能排名柱状图

In [ ]:
# 创建性能排名图
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图: 综合评分
top_6 = df_ranking.head(6)
colors = ['#2ecc71' if s > 3 else '#3498db' if s > 2 else '#f39c12' 
         for s in top_6['Performance Score']]

bars1 = axes[0].barh(range(len(top_6)), top_6['Performance Score'], color=colors,
                    edgecolor='black', linewidth=1.5)
axes[0].set_yticks(range(len(top_6)))
axes[0].set_yticklabels(top_6['Formula'], fontsize=12)
axes[0].set_xlabel('Performance Score', fontsize=14, fontweight='bold')
axes[0].set_title('Top 6 Candidates - Performance Score', 
                 fontsize=15, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# 添加数值
for i, (bar, score) in enumerate(zip(bars1, top_6['Performance Score'])):
    axes[0].text(score + 0.1, i, f'{score:.2f}', va='center', fontsize=11, fontweight='bold')

# 右图: 电导率对比
x = np.arange(len(top_6))
width = 0.35

bars2_1 = axes[1].bar(x - width/2, top_6['log10(σ_300K)'], width, 
                     label='300K', color='#3498db', edgecolor='black')
bars2_2 = axes[1].bar(x + width/2, np.log10(top_6['σ_500K (S/cm)']), width,
                     label='500K', color='#e74c3c', edgecolor='black')

axes[1].axhline(y=-4, color='green', linestyle='--', linewidth=2,
               label='Target (10⁻⁴ S/cm)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(top_6['Formula'], rotation=45, ha='right', fontsize=11)
axes[1].set_ylabel('log₁₀(σ) (S/cm)', fontsize=14, fontweight='bold')
axes[1].set_title('Ionic Conductivity Comparison', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{config.work_dir}/fig_ranking_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ 排名图生成完成")

## 12. 与实验值对比

In [ ]:
# 实验参考数据
experimental_data = {
    'Li3PS4': {'sigma': 3e-5, 'Ea': 0.25},
    'Li7P3S11': {'sigma': 1.7e-3, 'Ea': 0.18},
    'Li6PS5Cl': {'sigma': 1.5e-4, 'Ea': 0.22},
    'Li10GeP2S12': {'sigma': 1.2e-2, 'Ea': 0.25},
    'Li4GeS4': {'sigma': 5e-7, 'Ea': 0.45},
}

# 创建对比图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

materials = list(experimental_data.keys())
exp_sigmas = [experimental_data[m]['sigma'] for m in materials]
exp_eas = [experimental_data[m]['Ea'] for m in materials]

calc_sigmas = []
calc_eas = []

for m in materials:
    row = df_ranking[df_ranking['Formula'] == m]
    if not row.empty:
        calc_sigmas.append(row['σ_300K (S/cm)'].values[0])
        calc_eas.append(row['Activation Energy (eV)'].values[0])
    else:
        calc_sigmas.append(np.nan)
        calc_eas.append(np.nan)

x = np.arange(len(materials))
width = 0.35

# 左图: 电导率对比
axes[0].bar(x - width/2, np.log10(exp_sigmas), width, 
           label='Experimental', color='#e74c3c', edgecolor='black')
axes[0].bar(x + width/2, np.log10(calc_sigmas), width,
           label='Calculated', color='#3498db', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(materials, rotation=45, ha='right')
axes[0].set_ylabel('log₁₀(σ) at 300K (S/cm)', fontsize=14, fontweight='bold')
axes[0].set_title('Ionic Conductivity: Calc vs Exp', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# 右图: 活化能对比
axes[1].bar(x - width/2, exp_eas, width, 
           label='Experimental', color='#e74c3c', edgecolor='black')
axes[1].bar(x + width/2, calc_eas, width,
           label='Calculated', color='#3498db', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(materials, rotation=45, ha='right')
axes[1].set_ylabel('Activation Energy (eV)', fontsize=14, fontweight='bold')
axes[1].set_title('Activation Energy: Calc vs Exp', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{config.work_dir}/fig_validation_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ 验证图生成完成")

## 13. 生成报告

In [ ]:
# 生成详细报告
screener.generate_report(md_results, df_ranking)

print("\n报告文件:")
report_files = list(Path(config.work_dir).glob('*.txt'))
for f in report_files:
    print(f"  {f.name}")

print("\n结果文件:")
result_files = list(Path(config.work_dir).glob('*.csv'))
for f in result_files:
    print(f"  {f.name}")

## 14. 总结

In [ ]:
print("="*70)
print("固态电解质筛选案例研究 - 完成摘要")
print("="*70)
print(f"\n总候选材料: {len(candidates)}")
print(f"成功计算: {len(md_results)}")
print(f"\n最佳候选: {df_ranking.iloc[0]['Formula']}")
print(f"  - Material ID: {df_ranking.iloc[0]['Material ID']}")
print(f"  - Space Group: {df_ranking.iloc[0]['Space Group']}")
print(f"  - Activation Energy: {df_ranking.iloc[0]['Activation Energy (eV)']:.3f} eV")
print(f"  - σ at 300K: {df_ranking.iloc[0]['σ_300K (S/cm)']:.2e} S/cm")
print(f"  - Performance Score: {df_ranking.iloc[0]['Performance Score']:.2f}")
print(f"  - Recommendation: {df_ranking.iloc[0]['Recommendation']}")

print("\n推荐候选:")
for i, row in df_ranking.head(3).iterrows():
    print(f"  {i+1}. {row['Formula']}: {row['Recommendation']}")

print(f"\n所有结果保存在: {Path(config.work_dir).absolute()}")
print("="*70)